# Exploratory Data Analysis & Ensemble Experiments (Iris Dataset)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vantage-Consortium-Co-Ltd/Ensemble-Learning/blob/phuriphat/notebooks/01_eda_and_experiments.ipynb)

## 🎯 Notebook Objectives
1. **Exploratory Data Analysis (EDA)** on the Iris dataset: summary statistics, missing values, correlation analysis, and pairwise distributions.
2. **Weak Learner Mechanics**: Deep dive into Decision Stump splitting criteria and multi-class SAMME sample weight updates.
3. **Decision Boundary Visualizations**: Plotting 2D decision boundary surfaces comparing a Single Decision Tree vs. Bagging vs. AdaBoost.
4. **Variance vs. Bias Trade-off**: Empirical demonstration of how Bagging reduces variance while Boosting reduces bias.

### 🛠️ Cell 1: Environment Setup & Imports

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Detect environment and configure sys.path
is_colab = 'google.colab' in sys.modules or os.path.exists('/content')
if is_colab:
    !pip install --quiet datasets pyyaml scikit-learn pandas matplotlib tabulate joblib
    repo_path = "/content/Ensemble-Learning"
    if not os.path.exists(repo_path):
        !git clone -b phuriphat https://github.com/Vantage-Consortium-Co-Ltd/Ensemble-Learning.git {repo_path}
    if f"{repo_path}/src" not in sys.path:
        sys.path.insert(0, f"{repo_path}/src")
    os.chdir(repo_path)
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    repo_src = os.path.join(repo_root, "src")
    if repo_src not in sys.path:
        sys.path.insert(0, repo_src)

from ensemble_learning.data.make_dataset import load_iris_data, split_dataset
from ensemble_learning.models.bagging import BaggingClassifier
from ensemble_learning.models.boosting import AdaBoostClassifier
from ensemble_learning.utils.metrics import compute_classification_metrics

print("Environment initialized successfully.")


### 📊 Cell 2: Dataset Ingestion & Statistical Profiling
Load dataset from local cache or Hugging Face repository `antoniopelusi/iris`.

In [ ]:
df = load_iris_data(raw_data_dir="data/raw")
print("Dataset Shape:", df.shape)
print("\nMissing values per column:")
print(df.isnull().sum())

print("\nDescriptive Statistics:")
display(df.describe().T)


### 🔍 Cell 3: Pairwise Feature Relationships & Correlation Analysis

In [ ]:
feature_cols = ["sepal-length", "sepal-width", "petal-length", "petal-width"]
species_labels = ["Setosa", "Versicolor", "Virginica"]
colors = ["#1f77b4", "#2ca02c", "#d62728"]

# Correlation Matrix
corr = df[feature_cols].corr()
fig, ax = plt.subplots(figsize=(6, 5))
cax = ax.matshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
fig.colorbar(cax)
ax.set_xticks(range(len(feature_cols)))
ax.set_yticks(range(len(feature_cols)))
ax.set_xticklabels(feature_cols, rotation=45, ha="left")
ax.set_yticklabels(feature_cols)
for i in range(len(feature_cols)):
    for j in range(len(feature_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", color="black" if abs(corr.iloc[i, j]) < 0.6 else "white")
plt.title("Feature Correlation Matrix", pad=20, fontweight="bold")
plt.show()

# Pairwise Scatter Matrix (4x4)
fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for i, f1 in enumerate(feature_cols):
    for j, f2 in enumerate(feature_cols):
        ax = axes[i, j]
        if i == j:
            for c_id, (sp, col) in enumerate(zip(species_labels, colors)):
                vals = df[df["target"] == c_id][f1]
                ax.hist(vals, bins=10, alpha=0.5, color=col, edgecolor="black")
        else:
            for c_id, (sp, col) in enumerate(zip(species_labels, colors)):
                sub = df[df["target"] == c_id]
                ax.scatter(sub[f2], sub[f1], color=col, alpha=0.7, s=25)
        if i == 3:
            ax.set_xlabel(f2, fontsize=9)
        if j == 0:
            ax.set_ylabel(f1, fontsize=9)
        ax.grid(alpha=0.2)
plt.suptitle("Iris 4x4 Pairwise Scatter and Distribution Matrix", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


### 🌲 Cell 4: Weak Learner Mechanics & Multi-Class SAMME Breakdown
In AdaBoost SAMME (Hastie et al., 2009):
1. Each weak learner is a **Decision Stump** (Decision Tree with `max_depth=1`).
2. The weighted error is computed as:
   $$\text{err}_m = \frac{\sum_{i=1}^N w_i \cdot \mathbb{I}(y_i \neq T_m(x_i))}{\sum_{i=1}^N w_i}$$
3. The multi-class estimator weight $\alpha_m$ incorporates the number of classes $K$:
   $$\alpha_m = \log \frac{1 - \text{err}_m}{\text{err}_m} + \log(K - 1)$$
4. Sample weights are updated exponentially:
   $$w_i \leftarrow w_i \exp\big(\alpha_m \cdot \mathbb{I}(y_i \neq T_m(x_i))\big)$$
Let us observe the evolution of $\alpha_m$ and estimator errors across boosting rounds.

In [ ]:
X_train, X_test, y_train, y_test = split_dataset(df, train_ratio=0.8, test_ratio=0.2, random_state=42)

ada = AdaBoostClassifier(n_estimators=30, learning_rate=1.0, random_state=42)
ada.fit(X_train, y_train)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(range(1, len(ada.estimator_errors_) + 1), ada.estimator_errors_, marker="o", color="#d62728")
ax1.set_title("Estimator Weighted Error ($err_m$)", fontweight="bold")
ax1.set_xlabel("Boosting Iteration (m)")
ax1.set_ylabel("Weighted Error")
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, len(ada.estimator_weights_) + 1), ada.estimator_weights_, marker="s", color="#1f77b4")
ax2.set_title("Estimator Weight ($\alpha_m$)", fontweight="bold")
ax2.set_xlabel("Boosting Iteration (m)")
ax2.set_ylabel("Weight $\\alpha_m$")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 🎨 Cell 5: 2D Decision Boundary Comparison (Single Tree vs. Bagging vs. AdaBoost)
We train models on the two most discriminative features: `petal-length` and `petal-width`.
Comparing:
1. Single Decision Stump (`max_depth=1`)
2. Single Decision Tree (`max_depth=3`)
3. BaggingClassifier (`n_estimators=50`)
4. AdaBoostClassifier (`n_estimators=50`)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Extract 2 features for 2D visualization
X_2d_train = X_train[:, [2, 3]]
X_2d_test = X_test[:, [2, 3]]

models = {
    "Decision Stump (depth=1)": DecisionTreeClassifier(max_depth=1, random_state=42),
    "Decision Tree (depth=3)": DecisionTreeClassifier(max_depth=3, random_state=42),
    "Bagging (50 trees)": BaggingClassifier(n_estimators=50, random_state=42),
    "AdaBoost SAMME (50 stumps)": AdaBoostClassifier(n_estimators=50, random_state=42),
}

# Create mesh grid
x_min, x_max = X_2d_train[:, 0].min() - 0.5, X_2d_train[:, 0].max() + 0.5
y_min, y_max = X_2d_train[:, 1].min() - 0.5, X_2d_train[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
grid_points = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
cmap_light = ["#dbe9f6", "#d9f0d3", "#fcdbd9"]
from matplotlib.colors import ListedColormap
cm_bg = ListedColormap(cmap_light)

for ax, (name, clf) in zip(axes.flatten(), models.items()):
    clf.fit(X_2d_train, y_train)
    Z = clf.predict(grid_points).reshape(xx.shape)
    test_acc = np.mean(clf.predict(X_2d_test) == y_test)
    
    ax.contourf(xx, yy, Z, cmap=cm_bg, alpha=0.6)
    for c_id, (sp, col) in enumerate(zip(species_labels, colors)):
        idx = y_train == c_id
        ax.scatter(X_2d_train[idx, 0], X_2d_train[idx, 1], color=col, edgecolors="black", s=40, label=f"{sp} (Train)")
    ax.set_title(f"{name}\nTest Accuracy: {test_acc:.3f}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Petal Length (cm)")
    ax.set_ylabel("Petal Width (cm)")
    ax.grid(alpha=0.3)
    if name == "Decision Stump (depth=1)":
        ax.legend(loc="upper left", fontsize=8)

plt.tight_layout()
plt.show()


### 📈 Cell 6: Cumulative Estimator Accuracy Convergence
We examine how test accuracy evolves as additional estimators are incorporated into the ensemble.

In [ ]:
# AdaBoost staged predictions
staged_accs = [np.mean(y_stage == y_test) for y_stage in ada.staged_predict(X_test)]

plt.figure(figsize=(8, 4.5))
plt.plot(range(1, len(staged_accs) + 1), staged_accs, marker="o", color="#d62728", linewidth=2)
plt.title("AdaBoost SAMME: Test Accuracy by Iteration Stage", fontsize=12, fontweight="bold")
plt.xlabel("Boosting Stage / Estimator Count")
plt.ylabel("Test Accuracy")
plt.ylim([0.8, 1.02])
plt.grid(True, alpha=0.3)
plt.show()


### 💾 Cell 7: Model Export & Summary
Serializing final models and recording architectural takeaways:
- **Bagging**: Parallelizable, robust to noise and outliers, primarily reduces **variance**.
- **Boosting**: Sequential, rapidly drives down training error and **bias**, achieves high margins with minimal base learner depth.

In [ ]:
import joblib
from pathlib import Path

out_dir = Path("models")
out_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(ada, out_dir / "adaboost_eda_final.pkl")
print("✅ EDA and experiments workflow complete. Model saved.")
